In [20]:
# -*- coding: utf-8 -*-
import pandas as pd
import numpy as np
import sys
import os
from pathlib import Path
import importlib.util

import warnings
warnings.filterwarnings('ignore')

import logging
logging.basicConfig(level=logging.INFO)
logger = logging.getLogger(__name__)

# plotting 설정
import matplotlib
import matplotlib.pyplot as plt
matplotlib.rc('font', family='Malgun Gothic')
plt.rcParams['axes.unicode_minus'] = False

# from datetime import datetime, timedelta
from typing import Iterable
from sklearn.preprocessing import StandardScaler
from tqdm import tqdm
import requests

# stock_forecast 폴더로 디렉토리 변경
current_dir = Path.cwd()
stock_forecast_dir = current_dir.parent.parent  # 2단계 상위로 이동 (필요시 조정)
if stock_forecast_dir.name != 'stock_forecast':
    # 이미 stock_forecast 폴더에 있다면
    stock_forecast_dir = current_dir

os.chdir(stock_forecast_dir)

# DATA 폴더의 모든 Python 파일을 자동으로 import
data_path = stock_forecast_dir / 'DATA'
sys.path.insert(0, str(data_path))

# DATA 폴더 내의 모든 .py 파일 찾기
py_files = [f for f in data_path.glob('*.py') if f.name != '__init__.py']

print(f"DATA 폴더에서 발견된 Python 파일들:")
for py_file in py_files:
    print(f"  - {py_file.name}")

from datetime import datetime, timedelta

# 각 파일을 모듈로 import
imported_modules = {}
for py_file in py_files:
    try:
        module_name = py_file.stem  # 파일명에서 확장자 제거

        # 방법 1: 직접 import 시도
        try:
            module = __import__(module_name)
            imported_modules[module_name] = module

            # 모든 함수와 변수를 전역 네임스페이스로 가져오기
            for attr_name in dir(module):
                if not attr_name.startswith('_'):
                    globals()[attr_name] = getattr(module, attr_name)

            print(f"✅ {module_name} import 성공 (직접 import)")

        except ImportError:
            # 방법 2: importlib 사용
            spec = importlib.util.spec_from_file_location(module_name, py_file)
            module = importlib.util.module_from_spec(spec)
            spec.loader.exec_module(module)
            imported_modules[module_name] = module

            # 모든 함수와 변수를 전역 네임스페이스로 가져오기
            for attr_name in dir(module):
                if not attr_name.startswith('_'):
                    globals()[attr_name] = getattr(module, attr_name)

            print(f"✅ {module_name} import 성공 (importlib 사용)")

    except Exception as e:
        print(f"❌ {py_file.name} import 실패: {str(e)}")

# SQLAlchemy
from sqlalchemy import create_engine, text, Table, MetaData
from sqlalchemy.dialects.mysql import insert as mysql_insert

# import 구문을 이렇게 변경
import datetime
from datetime import timedelta


print(f"\n✅ 총 {len(imported_modules)}개 모듈 import 완료")
print(f"현재 디렉토리: {Path.cwd()}")
print(f"import된 모듈들: {list(imported_modules.keys())}")

# 사용 가능한 함수들 확인 (선택사항)
print(f"\n주요 함수들:")
all_functions = [name for name in globals().keys() if callable(globals()[name]) and not name.startswith('_')]
main_functions = [f for f in all_functions if any(keyword in f.lower() for keyword in ['fetch', 'get', 'load', 'save', 'process'])]
for func in sorted(main_functions)[:10]:  # 처음 10개만 표시
    print(f"  - {func}")
if len(main_functions) > 10:
    print(f"  ... 및 {len(main_functions)-10}개 더")

DATA 폴더에서 발견된 Python 파일들:
  - ANET_Valuation.py
  - stock_invest_function.py
✅ ANET_Valuation import 성공 (직접 import)
✅ stock_invest_function import 성공 (직접 import)

✅ 총 2개 모듈 import 완료
현재 디렉토리: C:\Users\Hoyoung_Park\PyCharmMiscProject\stock_forecast
import된 모듈들: ['ANET_Valuation', 'stock_invest_function']

주요 함수들:
  - fetch_table_data
  - fetch_trade_data_multi_hscode
  - get_FMP_data
  - get_data_by_ticker
  - get_db_host
  - get_ipython
  - get_market_cap
  - get_market_cap_by_ticker
  - get_quarterly_export_forecast
  - load_forecast_by_hscode
  ... 및 2개 더


In [22]:
# -----------------------------
# Parameters
# -----------------------------
tic_name = 'ANET'                # 티커
hs_code = '851762'             # 대외 변수 HS CODE (옵션)
item_name = 'PSR'
st_date = '2010-01-01'
end_date = '2025-07-31'
# 그러면 원래 코드가 작동합니다
today_date = pd.to_datetime(datetime.datetime.today().date())

USE_EXOGENOUS = True             # 외생변수 사용 여부

apikey = 'hT0gAk87j9xZx4PlBApvBqfVL5IahvgV'

db_info = {
    'host': get_db_host(),
    'port': 3307,
    'user': 'stox7412',
    'password': 'Apt106503!~',
    'database': 'investar'
}

#### 1. 재무데이터 입력하기

In [23]:
#!/usr/bin/env python3
# -*- coding: utf-8 -*-
"""
Financial Modeling Prep API - 분기별 매출 데이터 수집 스크립트
10년(40분기) 완전한 데이터가 있는 기업만 수집 + 결측치 처리
"""

import requests
import pandas as pd
from datetime import datetime
import time
import numpy as np

# ==============================================
# 설정값들
# ==============================================

# API 키 설정 (https://financialmodelingprep.com/developer/docs 에서 발급)
API_KEY =  apikey  # 실제 API 키로 변경하세요

# 수집할 티커들 (대형 기업들 - 10년 이상 데이터 보장)
TICKERS = [
    'AAPL', 'MSFT', 'GOOGL', 'AMZN', 'META',
    'TSLA', 'NVDA', 'NFLX', 'AMD', 'INTC',
    'CRM', 'ORCL', 'ADBE', 'PYPL', 'UBER',
    'DIS', 'NKE', 'SBUX', 'MCD', 'KO',
    'PEP', 'WMT', 'TGT', 'HD', 'LOW',
    'JNJ', 'PFE', 'UNH', 'ABBV', 'MRK',
    'JPM', 'BAC', 'WFC', 'GS', 'MS',
    'V', 'MA', 'AXP', 'COF', 'SCHW',
    'IBM', 'CSCO', 'QCOM', 'BKNG', 'ISRG',
    'TMO', 'LLY', 'CVX', 'XOM', 'RTX'
]

# 정확히 10년 = 40분기 데이터 요구
REQUIRED_QUARTERS = 40

# 재시도 설정
MAX_RETRIES = 2  # 최대 재시도 횟수
REQUEST_DELAY = 0.3  # API 요청 간 지연시간 (초)
RETRY_DELAY = 1.0    # 재시도 간 지연시간 (초)

# ==============================================
# 유틸리티 함수들
# ==============================================

def check_missing_data(data_list, ticker):
    """분기별 데이터에서 결측치 확인"""
    missing_info = []

    for i, item in enumerate(data_list):
        issues = []

        # 매출이 None이거나 0인 경우
        if item.get('revenue') is None or item.get('revenue') == 0:
            issues.append('revenue_missing')

        # 날짜가 없는 경우
        if not item.get('date'):
            issues.append('date_missing')

        # 분기 정보가 없는 경우
        if not item.get('period'):
            issues.append('period_missing')

        if issues:
            missing_info.append({
                'ticker': ticker,
                'quarter_index': i,
                'date': item.get('date', 'Unknown'),
                'period': item.get('period', 'Unknown'),
                'issues': ', '.join(issues),
                'revenue': item.get('revenue', 'N/A')
            })

    return missing_info

def fetch_ticker_data(ticker, retry_count=0):
    """개별 티커 데이터 수집 (재시도 포함)"""

    url = f"https://financialmodelingprep.com/api/v3/income-statement/{ticker}"
    params = {
        'limit': 50,  # 여유분 포함
        'apikey': API_KEY,
        'period': 'quarter'
    }

    try:
        response = requests.get(url, params=params, timeout=30)
        response.raise_for_status()
        data = response.json()

        if not data:
            return None, f"No data returned"

        if len(data) < REQUIRED_QUARTERS:
            return None, f"Insufficient data ({len(data)} quarters, need {REQUIRED_QUARTERS})"

        return data, None

    except requests.exceptions.Timeout:
        return None, f"Request timeout (attempt {retry_count + 1})"
    except requests.exceptions.RequestException as e:
        return None, f"Request failed: {str(e)} (attempt {retry_count + 1})"
    except Exception as e:
        return None, f"Unexpected error: {str(e)} (attempt {retry_count + 1})"

# ==============================================
# API 키 확인
# ==============================================
if API_KEY == "YOUR_API_KEY_HERE":
    print("❌ API 키를 설정해주세요!")
    print("https://financialmodelingprep.com/developer/docs 에서 무료 API 키를 발급받을 수 있습니다.")
    exit()

print(f"🚀 10년 분기별 매출 데이터 수집 시작 (정확히 {REQUIRED_QUARTERS}분기)")
print(f"📊 대상 기업: {len(TICKERS)}개")
print(f"🔄 재시도 설정: 최대 {MAX_RETRIES}회")
print("=" * 70)

# ==============================================
# 데이터 수집
# ==============================================
all_revenue_data = []
successful_tickers = []
failed_tickers = []
missing_data_tickers = []  # 결측치 있는 기업들
all_missing_info = []      # 모든 결측치 정보

for i, ticker in enumerate(TICKERS):
    print(f"\n[{i+1:2d}/{len(TICKERS)}] {ticker} 데이터 수집 중...")

    success = False
    data = None
    last_error = ""

    # 재시도 로직
    for retry in range(MAX_RETRIES + 1):
        if retry > 0:
            print(f"   🔄 재시도 {retry}/{MAX_RETRIES}...")
            time.sleep(RETRY_DELAY)

        data, error = fetch_ticker_data(ticker, retry)

        if data is not None:
            success = True
            break
        else:
            last_error = error
            print(f"   ⚠️  {error}")

    if not success:
        print(f"   ❌ {ticker}: 최종 실패 - {last_error}")
        failed_tickers.append({'ticker': ticker, 'reason': last_error})
        continue

    # 결측치 확인
    missing_info = check_missing_data(data[:REQUIRED_QUARTERS], ticker)

    if missing_info:
        print(f"   ⚠️  {ticker}: 결측치 발견 ({len(missing_info)}개)")
        missing_data_tickers.append(ticker)
        all_missing_info.extend(missing_info)

        # 결측치가 있어도 데이터가 충분하면 포함할지 결정
        valid_data_count = sum(1 for item in data[:REQUIRED_QUARTERS]
                              if item.get('revenue') is not None and item.get('revenue') > 0)

        if valid_data_count < REQUIRED_QUARTERS * 0.9:  # 90% 이하면 제외
            print(f"   ❌ {ticker}: 결측치 너무 많음 (유효 데이터 {valid_data_count}개)")
            failed_tickers.append({'ticker': ticker, 'reason': f'Too many missing values ({valid_data_count}/{REQUIRED_QUARTERS})'})
            continue

    # 데이터 파싱 (정확히 40분기)
    quarterly_data = []
    for j, item in enumerate(data[:REQUIRED_QUARTERS]):
        quarterly_data.append({
            'ticker': ticker,
            'quarter_sequence': j + 1,  # 1-40 순서
            'date': item.get('date', ''),
            'calendar_year': item.get('calendarYear', ''),
            'period': item.get('period', ''),
            'revenue': item.get('revenue', 0) if item.get('revenue') is not None else 0,
            'revenue_millions': round((item.get('revenue', 0) or 0) / 1_000_000, 1),
            'revenue_billions': round((item.get('revenue', 0) or 0) / 1_000_000_000, 2),
            'gross_profit': item.get('grossProfit', 0) if item.get('grossProfit') is not None else 0,
            'gross_margin': round(((item.get('grossProfit', 0) or 0) / (item.get('revenue', 1) or 1)) * 100, 2) if (item.get('revenue') or 0) > 0 else 0,
            'reported_currency': item.get('reportedCurrency', 'USD'),
            'has_missing_data': len([info for info in missing_info if info['quarter_index'] == j]) > 0
        })

    # 전체 데이터에 추가
    all_revenue_data.extend(quarterly_data)
    successful_tickers.append(ticker)

    # 결과 출력
    latest = quarterly_data[0]
    missing_note = f" (결측치 {len(missing_info)}개)" if missing_info else ""
    print(f"   ✅ {ticker}: {len(quarterly_data)}분기, 최신 매출 ${latest['revenue_billions']}B ({latest['date']}){missing_note}")

    # API 제한 대응을 위한 지연
    if i < len(TICKERS) - 1:
        time.sleep(REQUEST_DELAY)

# ==============================================
# 결과 정리 및 출력
# ==============================================
print("\n" + "=" * 70)
print(f"📈 데이터 수집 완료!")
print(f"✅ 성공: {len(successful_tickers)}개 기업")
print(f"❌ 실패: {len(failed_tickers)}개 기업")
print(f"⚠️  결측치 발견: {len(missing_data_tickers)}개 기업")

# 실패한 기업들 상세 정보
if failed_tickers:
    print(f"\n❌ 실패한 기업들:")
    for fail_info in failed_tickers:
        print(f"   - {fail_info['ticker']}: {fail_info['reason']}")

# 결측치 있는 기업들
if missing_data_tickers:
    print(f"\n⚠️  결측치가 있는 기업들:")
    for ticker in missing_data_tickers:
        ticker_missing = [info for info in all_missing_info if info['ticker'] == ticker]
        print(f"   - {ticker}: {len(ticker_missing)}개 분기에서 결측치")

if not all_revenue_data:
    print("❌ 수집된 데이터가 없습니다.")
    exit()

# DataFrame 생성
df = pd.DataFrame(all_revenue_data)
df['date'] = pd.to_datetime(df['date'])
df = df.sort_values(['ticker', 'date'], ascending=[True, False])

# 기본 통계 출력
print(f"\n📊 최종 수집 데이터:")
print(f"   - 총 레코드: {len(df):,}개")
print(f"   - 기간: {df['date'].min().strftime('%Y-%m-%d')} ~ {df['date'].max().strftime('%Y-%m-%d')}")
print(f"   - 총 매출 합계: ${df['revenue_billions'].sum():,.0f}B")
print(f"   - 평균 분기 매출: ${df['revenue_billions'].mean():.2f}B")

# ==============================================
# 최종 DataFrame 확인
# ==============================================
print(f"\n📋 최종 DataFrame 구조:")
print(f"   - Shape: {df.shape}")
print(f"   - Columns: {list(df.columns)}")

print(f"\n📊 DataFrame 정보:")
print(df.info())

print(f"\n📈 최종 데이터 샘플 (상위 5행):")
print("=" * 100)
pd.set_option('display.max_columns', None)
pd.set_option('display.width', None)
pd.set_option('display.max_colwidth', 20)
print(df.head(5))

print(f"\n📊 기업별 데이터 요약:")
print("=" * 70)
summary = df.groupby('ticker').agg({
    'date': ['count', 'min', 'max'],
    'revenue_billions': ['sum', 'mean', 'std'],
    'has_missing_data': 'sum'
}).round(2)
summary.columns = ['분기수', '시작일', '종료일', '총매출(B)', '평균매출(B)', '매출표준편차', '결측분기수']
print(summary.head(10))

# 실패한 기업 정보 출력
if failed_tickers:
    print(f"\n❌ 실패한 기업 상세:")
    failed_df = pd.DataFrame(failed_tickers)
    print(failed_df)

# 결측치 정보 출력
if all_missing_info:
    print(f"\n⚠️  결측치 상세 정보 (최대 10개):")
    missing_df = pd.DataFrame(all_missing_info)
    print(missing_df.head(10))

# ==============================================
# 최종 데이터 품질 리포트
# ==============================================
print("\n" + "=" * 70)
print("📋 데이터 품질 리포트")
print("=" * 70)

# 매출 상위 5개 기업
print(f"🏆 매출 상위 5개 기업 (10년 누적):")
top5 = df.groupby('ticker')['revenue_billions'].sum().nlargest(5)
for rank, (ticker, total_revenue) in enumerate(top5.items(), 1):
    quarters_with_data = len(df[(df['ticker'] == ticker) & (df['revenue'] > 0)])
    print(f"   {rank}. {ticker}: ${total_revenue:,.0f}B ({quarters_with_data}/40 분기 데이터)")

# 데이터 완전성 체크
complete_data_tickers = [ticker for ticker in successful_tickers if ticker not in missing_data_tickers]
print(f"\n📊 데이터 완전성:")
print(f"   - 완전한 데이터 (결측치 없음): {len(complete_data_tickers)}개 기업")
print(f"   - 일부 결측치 있음: {len(missing_data_tickers)}개 기업")
print(f"   - 수집 실패: {len(failed_tickers)}개 기업")

if complete_data_tickers:
    print(f"\n✨ 완전한 데이터를 가진 기업들:")
    print(f"   {', '.join(complete_data_tickers[:10])}" + ("..." if len(complete_data_tickers) > 10 else ""))

print(f"\n🎉 10년 분기별 매출 데이터 수집 완료!")
print(f"📈 최종 수집: {len(successful_tickers)}개 기업, {len(df):,}개 분기 데이터")

# DataFrame을 전역 변수로 사용 가능하도록 저장
print(f"\n💾 데이터는 'df' 변수에 저장되어 있습니다.")
print(f"   - df.shape: {df.shape}")
print(f"   - 사용 예시: df[df['ticker'] == 'AAPL'].head()")
print(f"   - 전체 기업 목록: {sorted(df['ticker'].unique())}")

🚀 10년 분기별 매출 데이터 수집 시작 (정확히 40분기)
📊 대상 기업: 50개
🔄 재시도 설정: 최대 2회

[ 1/50] AAPL 데이터 수집 중...
   ✅ AAPL: 40분기, 최신 매출 $94.04B (2025-06-28)

[ 2/50] MSFT 데이터 수집 중...
   ✅ MSFT: 40분기, 최신 매출 $76.44B (2025-06-30)

[ 3/50] GOOGL 데이터 수집 중...
   ✅ GOOGL: 40분기, 최신 매출 $96.43B (2025-06-30)

[ 4/50] AMZN 데이터 수집 중...
   ✅ AMZN: 40분기, 최신 매출 $167.7B (2025-06-30)

[ 5/50] META 데이터 수집 중...
   ✅ META: 40분기, 최신 매출 $47.52B (2025-06-30)

[ 6/50] TSLA 데이터 수집 중...
   ✅ TSLA: 40분기, 최신 매출 $22.5B (2025-06-30)

[ 7/50] NVDA 데이터 수집 중...
   ✅ NVDA: 40분기, 최신 매출 $46.74B (2025-07-27)

[ 8/50] NFLX 데이터 수집 중...
   ✅ NFLX: 40분기, 최신 매출 $11.08B (2025-06-30)

[ 9/50] AMD 데이터 수집 중...
   ✅ AMD: 40분기, 최신 매출 $7.68B (2025-06-28)

[10/50] INTC 데이터 수집 중...
   ✅ INTC: 40분기, 최신 매출 $12.86B (2025-06-28)

[11/50] CRM 데이터 수집 중...
   ✅ CRM: 40분기, 최신 매출 $10.24B (2025-07-31)

[12/50] ORCL 데이터 수집 중...
   ✅ ORCL: 40분기, 최신 매출 $14.93B (2025-08-31)

[13/50] ADBE 데이터 수집 중...
   ✅ ADBE: 40분기, 최신 매출 $5.87B (2025-05-30)

[14/50] PYPL 데이터 수집 중...
   ✅ P

In [4]:
df

,ticker,quarter_sequence,date,calendar_year,period,revenue,revenue_millions,revenue_billions,gross_profit,gross_margin,reported_currency,has_missing_data
0,AAPL,1,2025-06-28,2025,Q3,94036000000,94036.0,94.04,43718000000,46.49,USD,False
1,AAPL,2,2025-03-29,2025,Q2,95359000000,95359.0,95.36,44867000000,47.05,USD,False
2,AAPL,3,2024-12-28,2025,Q1,124300000000,124300.0,124.30,58275000000,46.88,USD,False
3,AAPL,4,2024-09-28,2024,Q4,94930000000,94930.0,94.93,43879000000,46.22,USD,False
4,AAPL,5,2024-06-29,2024,Q3,85777000000,85777.0,85.78,39678000000,46.26,USD,False
...,...,...,...,...,...,...,...,...,...,...,...,...
1915,XOM,36,2016-09-30,2016,Q3,56767000000,56767.0,56.77,10981000000,19.34,USD,False
1916,XOM,37,2016-06-30,2016,Q2,56360000000,56360.0,56.36,10898000000,19.34,USD,False
1917,XOM,38,2016-03-31,2016,Q1,47105000000,47105.0,47.10,9257000000,19.65,USD,False
1918,XOM,39,2015-12-31,2015,Q4,57691000000,57691.0,57.69,10841000000,18.79,USD,False
